# 06 — Threading & the GIL
**References:** Ramalho *Fluent Python* Ch. 19 · Beazley *Python Cookbook* Ch. 12 · PEP 703 (no-GIL)

## Narrative thread
```
Threading basics -> GIL explained -> I/O vs CPU-bound -> Thread safety -> concurrent.futures -> multiprocessing
```

## 1. The GIL (Global Interpreter Lock)

CPython's **GIL** is a mutex that allows only one thread to execute Python bytecode at a time. This means:

- **I/O-bound work:** threads ARE effective — GIL is released during blocking I/O, sleep, and C extensions (numpy, etc.)
- **CPU-bound work:** threads are NOT effective for pure Python — use `multiprocessing` or free-threaded Python (3.13, PEP 703)

**GIL release points:**
- Every 5ms (switchinterval) the GIL is released to allow thread switching
- During any system call (file I/O, socket I/O, `time.sleep`)
- During C extension calls that release it (`numpy`, `ctypes`, etc.)

| Task type | Use |
|---|---|
| I/O-bound | `threading` or `asyncio` |
| CPU-bound Python | `multiprocessing` or `ProcessPoolExecutor` |
| CPU-bound C extension | `threading` (extension releases GIL) |

In [ ]:
import threading
import time
import queue
import concurrent.futures
import urllib.request
from collections import Counter

# ── GIL: threads vs processes for CPU-bound work ─────────────────────────
def cpu_work(n):
    """Pure Python CPU work: sum of range."""
    return sum(range(n))

N, WORKERS = 5_000_000, 4

# Sequential
t0 = time.perf_counter()
results = [cpu_work(N) for _ in range(WORKERS)]
t_seq = time.perf_counter() - t0

# Threading (GIL prevents true parallelism for CPU-bound)
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=WORKERS) as ex:
    results_threads = list(ex.map(cpu_work, [N]*WORKERS))
t_thread = time.perf_counter() - t0

# Multiprocessing (bypasses GIL)
t0 = time.perf_counter()
with concurrent.futures.ProcessPoolExecutor(max_workers=WORKERS) as ex:
    results_procs = list(ex.map(cpu_work, [N]*WORKERS))
t_proc = time.perf_counter() - t0

print(f'CPU-bound ({WORKERS} tasks, n={N:,})')
print(f'  Sequential:       {t_seq:.3f} s  (baseline)')
print(f'  ThreadPool:       {t_thread:.3f} s  ({t_thread/t_seq:.2f}x) — GIL prevents speedup')
print(f'  ProcessPool:      {t_proc:.3f} s  ({t_proc/t_seq:.2f}x) — true parallelism')

In [ ]:
# ── Thread safety: race conditions and locks ──────────────────────────────
class UnsafeCounter:
    def __init__(self):
        self.value = 0

    def increment(self, n=1000):
        for _ in range(n):          # read-increment-write is NOT atomic
            self.value += 1

class SafeCounter:
    def __init__(self):
        self.value = 0
        self._lock = threading.Lock()

    def increment(self, n=1000):
        for _ in range(n):
            with self._lock:        # holds the lock for one increment
                self.value += 1

def run_threads(counter, n_threads=10, increments=1000):
    threads = [threading.Thread(target=counter.increment, args=(increments,))
               for _ in range(n_threads)]
    for t in threads: t.start()
    for t in threads: t.join()
    return counter.value

expected = 10 * 1000
# Unsafe: often loses updates due to race
unsafe_vals = [run_threads(UnsafeCounter()) for _ in range(5)]
safe_vals   = [run_threads(SafeCounter())   for _ in range(5)]

print(f'Expected:             {expected}')
print(f'Unsafe counter vals:  {unsafe_vals}  (may have lost updates)')
print(f'Safe counter vals:    {safe_vals}   (always correct)')

print()
# ── Thread-local storage ─────────────────────────────────────────────────
local = threading.local()

def worker(name):
    local.name = name                # each thread has its own .name
    time.sleep(0.01)
    return local.name                # reads ITS OWN value

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(worker, n): n for n in ['Alice', 'Bob', 'Carol']}
    for f, expected_name in futures.items():
        result = f.result()
        print(f'  Thread for {expected_name}: got {result!r} (correct: {result == expected_name})')

In [ ]:
# ── concurrent.futures: high-level API ───────────────────────────────────
def simulate_io(url_id: int) -> dict:
    """Simulate I/O-bound work (like HTTP request)."""
    time.sleep(0.05)  # simulated latency
    return {'id': url_id, 'status': 200, 'size': url_id * 100}

ids = list(range(20))

# Sequential
t0 = time.perf_counter()
[simulate_io(i) for i in ids]
t_seq = time.perf_counter() - t0

# ThreadPoolExecutor (ideal for I/O)
t0 = time.perf_counter()
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as ex:
    futures = [ex.submit(simulate_io, i) for i in ids]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]
t_thread = time.perf_counter() - t0

print(f'I/O-bound ({len(ids)} tasks, 50ms each)')
print(f'  Sequential: {t_seq:.3f} s')
print(f'  Threads:    {t_thread:.3f} s  ({t_seq/t_thread:.1f}x speedup)')

print()
# ── Cancellation and exception handling ──────────────────────────────────
def maybe_fail(n):
    if n == 5: raise ValueError(f'Failed on {n}')
    return n * 2

with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
    futs = {ex.submit(maybe_fail, i): i for i in range(8)}
    for fut in concurrent.futures.as_completed(futs):
        i = futs[fut]
        try:
            print(f'  {i} -> {fut.result()}')
        except ValueError as e:
            print(f'  {i} -> ERROR: {e}')

## 2. Synchronization primitives

| Primitive | Use case |
|---|---|
| `threading.Lock` | Mutual exclusion — only one thread enters |
| `threading.RLock` | Reentrant lock — same thread can acquire multiple times |
| `threading.Event` | Signal between threads — one-shot flag |
| `threading.Semaphore` | Limit concurrent access to a resource |
| `threading.Barrier` | Wait for all threads to reach a point |
| `queue.Queue` | Thread-safe FIFO — preferred for producer/consumer |

## 3. Key takeaways

| Concept | Rule |
|---|---|
| GIL | Released during I/O and C extensions; NOT released between Python bytecodes |
| I/O-bound | Use `ThreadPoolExecutor` or `asyncio` |
| CPU-bound | Use `ProcessPoolExecutor` or `multiprocessing` |
| Race condition | Any read-modify-write needs a `Lock` |
| `concurrent.futures` | High-level — prefer over raw threads/processes |
| `queue.Queue` | Preferred communication channel between threads |

**Next:** notebook 07 — async programming: asyncio, coroutines, and the event loop.